# Spring Gamma v3：三周数据质量与消融研究

固定窗口 2026-07-03–2026-07-23；exclusive cutoff `2026-07-24T00:00:00Z`。

## tl;dr

- RTH 不是零信号：92 个方向 origins，15/30/60m 标签为 74/71/60；但只有 7 个方向日。
- RTH/GTH exposure 关系不同，必须拆 session。
- RTH 60m Gamma/Charm WF ΔR² 为 7.93% / 11.17%，仅 3 个测试日。
- GTH 60m net-GEX WF ΔR² 为 4.13%。
- Raw-overlap tenor audit: 79 signals, 58 paired entries, 44/42/40 matched fixed exits.
- >=13:00 ET 只有 8/7/6 个 matched outcomes，三个 paired CI 全跨 0；不能证明 13:00 硬切换。
- Charm 只作 gate；Spring Gamma v3 保持 shadow-only。

## Context & Methods

报告方向标签按同交易日、目标 ±180 秒、ES 来源兼容联结。Greek 用 `as_of <= origin`，IV 必须用 `created_at <= origin`；expiry 必须一致。Greek/IV latest-valid 最大年龄分别为 300/600 秒。消融用前 4 个日期训练、之后逐日扩展；没有随机切分。

期限反事实固定报告 direction/wall，两个 tenor 使用同 provider/right/strike；entry 最新 <=15.00s live quote 按 ask，15/30/60m exit 最新 <=15.00s quote 按 bid。任何缺腿对称删除，不插值。

In [1]:
from scripts.build_spring_gamma_v3_research import build_research_payload, compact_table

payload = build_research_payload()
payload['metadata']

{'title': 'Spring Gamma v3：三周数据质量与消融研究',
 'window_start': '2026-07-03',
 'window_end': '2026-07-23',
 'cutoff_at': '2026-07-24T00:00:00Z',
 'timezone': 'America/New_York',
 'greek_max_age_seconds': 300.0,
 'iv_max_age_seconds': 600.0,
 'reports': 521,
 'analysis_dependencies': 'DuckDB point-in-time quote audit + NumPy + Python standard library',
 'status': 'shadow_only'}

## Matched 0DTE versus next-listed-expiry counterfactual

正的 `0DTE−1DTE` 才表示 0DTE 较好。置信区间按完整日期 bootstrap；单份 SPX 合约的美元 P&L 为点数乘 100，未计手续费与实际 fill。

In [2]:
print(payload['tenor_summary'])
print(compact_table(
    payload['tenor_coverage'],
    ['cutoff','horizon','eligible_signals','paired_entries','matched_exits','entry_coverage_pct','signal_coverage_pct','days'],
))

{'eligible_signals': 79, 'raw_overlap_days': 6, 'paired_entries': 58, 'matched_15m': 44, 'matched_30m': 42, 'matched_60m': 40}
cutoff     | horizon | eligible_signals | paired_entries | matched_exits | entry_coverage_pct | signal_coverage_pct | days
-----------+---------+------------------+----------------+---------------+--------------------+---------------------+-----
All RTH    | 15m     | 79               | 58             | 44            | 73.42              | 55.70               | 5   
All RTH    | 30m     | 79               | 58             | 42            | 73.42              | 53.16               | 5   
All RTH    | 60m     | 79               | 58             | 40            | 73.42              | 50.63               | 5   
<13:00 ET  | 15m     | 70               | 50             | 36            | 71.43              | 51.43               | 5   
<13:00 ET  | 30m     | 70               | 50             | 35            | 71.43              | 50.00               | 5   
<13:00 ET  |

In [3]:
print(compact_table(
    payload['tenor_performance'],
    ['cutoff','horizon','tenor','n','days','mean_pnl_points','median_pnl_points','win_rate_pct','median_entry_ask_points','median_entry_spread_points','mean_ci_low_points','mean_ci_high_points'],
))

cutoff     | horizon | tenor | n  | days | mean_pnl_points | median_pnl_points | win_rate_pct | median_entry_ask_points | median_entry_spread_points | mean_ci_low_points | mean_ci_high_points
-----------+---------+-------+----+------+-----------------+-------------------+--------------+-------------------------+----------------------------+--------------------+--------------------
All RTH    | 15m     | 0DTE  | 44 | 5    | -0.89           | -0.10             | 29.55        | 7.10                    | 0.10                       | -2.77              | 0.16               
All RTH    | 15m     | 1DTE  | 44 | 5    | -0.78           | -0.30             | 38.64        | 21.60                   | 0.20                       | -2.51              | 0.23               
All RTH    | 30m     | 0DTE  | 42 | 5    | -0.67           | -0.48             | 26.19        | 6.80                    | 0.10                       | -1.44              | 0.31               
All RTH    | 30m     | 1DTE  | 42 | 5   

In [4]:
print(compact_table(
    payload['tenor_paired_effect'],
    ['cutoff','horizon','n','days','mean_0dte_minus_1dte_points','median_0dte_minus_1dte_points','zero_better_pct','mean_ci_low_points','mean_ci_high_points','decision'],
))

cutoff     | horizon | n  | days | mean_0dte_minus_1dte_points | median_0dte_minus_1dte_points | zero_better_pct | mean_ci_low_points | mean_ci_high_points | decision              
-----------+---------+----+------+-----------------------------+-------------------------------+-----------------+--------------------+---------------------+-----------------------
All RTH    | 15m     | 44 | 5    | -0.11                       | -0.10                         | 45.45           | -0.33              | 0.09                | Inconclusive          
All RTH    | 30m     | 42 | 5    | -0.19                       | -0.18                         | 42.86           | -0.51              | 0.06                | Inconclusive          
All RTH    | 60m     | 40 | 5    | -0.79                       | -0.70                         | 32.50           | -1.26              | -0.30               | 1DTE favored in sample
<13:00 ET  | 15m     | 36 | 5    | -0.15                       | -0.10                         

In [5]:
assert payload['tenor_summary'] == {
    'eligible_signals': 79,
    'raw_overlap_days': 6,
    'paired_entries': 58,
    'matched_15m': 44,
    'matched_30m': 42,
    'matched_60m': 40,
}
post = [row for row in payload['tenor_paired_effect'] if row['cutoff'] == '>=13:00 ET']
assert [row['n'] for row in post] == [8, 7, 6]
assert all(row['mean_ci_low_points'] <= 0 <= row['mean_ci_high_points'] for row in post)
print('VALIDATED: paired raw-lineage quote coverage; no post-13:00 CI excludes zero.')

VALIDATED: paired raw-lineage quote coverage; no post-13:00 CI excludes zero.


## Daily walk-forward and separate expiry-touch path metric

Walk-forward 只用更早日期选择 tenor。到墙触达仅为 SPX 至 16:00 ET 的 realized path crossing，不是风险中性概率，也不进入固定窗口 P&L。

In [6]:
print(compact_table(
    payload['tenor_walk_forward'],
    ['cutoff','horizon','test_date','train_days','train_n','selected_tenor','test_n','test_mean_points','test_win_pct'],
))
print(compact_table(
    payload['expiry_touch'],
    ['cutoff','directional_wall_targets','path_covered','touch_rate_pct','touch_ci_low_pct','touch_ci_high_pct'],
))

cutoff     | horizon | test_date  | train_days | train_n | selected_tenor | test_n | test_mean_points | test_win_pct
-----------+---------+------------+------------+---------+----------------+--------+------------------+-------------
<13:00 ET  | 15m     | 2026-07-17 | 1          | 9       | 0DTE           | 7      | -5.83            | 14.29       
<13:00 ET  | 15m     | 2026-07-21 | 2          | 16      | 1DTE           | 5      | 0.58             | 60.00       
<13:00 ET  | 15m     | 2026-07-22 | 3          | 21      | 1DTE           | 9      | 0.22             | 44.44       
<13:00 ET  | 15m     | 2026-07-23 | 4          | 30      | 1DTE           | 6      | 0.47             | 66.67       
<13:00 ET  | 30m     | 2026-07-17 | 1          | 10      | 1DTE           | 6      | -1.40            | 33.33       
<13:00 ET  | 30m     | 2026-07-21 | 2          | 16      | 1DTE           | 5      | -0.02            | 40.00       
<13:00 ET  | 30m     | 2026-07-22 | 3          | 21      | 1DTE 

## Data

All-ES 与 directional-headline cohort 分开。后者用于 continuation gate；前者用于 raw ΔES 负对照。

In [7]:
print(compact_table(
    payload['cohort_summary'],
    ['cohort','session','origins','days','labels_15m','labels_30m','labels_60m'],
))

cohort               | session | origins | days | labels_15m | labels_30m | labels_60m
---------------------+---------+---------+------+------------+------------+-----------
All ES origins       | RTH     | 129     | 9    | 104        | 97         | 84        
All ES origins       | GTH     | 392     | 8    | 378        | 372        | 366       
Directional headline | RTH     | 92      | 7    | 74         | 71         | 60        
Directional headline | GTH     | 250     | 8    | 245        | 246        | 239       


In [8]:
coverage_focus = [row for row in payload['coverage'] if row['cohort'] == 'Directional headline']
print(compact_table(
    coverage_focus,
    ['session','feature','available','origins','coverage_pct','ci_low_pct','ci_high_pct'],
))

session | feature               | available | origins | coverage_pct | ci_low_pct | ci_high_pct
--------+-----------------------+-----------+---------+--------------+------------+------------
RTH     | Net GEX proxy         | 87        | 92      | 94.57        | 87.90      | 97.66      
RTH     | Gross Gamma           | 89        | 92      | 96.74        | 90.85      | 98.88      
RTH     | Gross Charm 5m        | 89        | 92      | 96.74        | 90.85      | 98.88      
RTH     | Gross Vanna 1vol      | 89        | 92      | 96.74        | 90.85      | 98.88      
RTH     | Zero-gamma structure  | 92        | 92      | 100.00       | 95.99      | 100.00     
RTH     | Put/Call skew         | 92        | 92      | 100.00       | 95.99      | 100.00     
RTH     | Put/Call walls        | 92        | 92      | 100.00       | 95.99      | 100.00     
RTH     | Usable contract ratio | 91        | 92      | 98.91        | 94.10      | 99.81      
GTH     | Net GEX proxy         | 203   

In [9]:
print(compact_table(
    payload['segment_coverage'],
    ['segment_et','origins','gex_pct','gamma_pct','charm_pct','vanna_pct','zero_gamma_pct','skew_pct'],
))

segment_et            | origins | gex_pct | gamma_pct | charm_pct | vanna_pct | zero_gamma_pct | skew_pct
----------------------+---------+---------+-----------+-----------+-----------+----------------+---------
Closed 16:00–20:15    | 2       | 0.00    | 0.00      | 0.00      | 0.00      | 0.00           | 0.00    
Overnight 20:15–03:00 | 124     | 86.29   | 70.97     | 70.97     | 70.97     | 93.55          | 89.52   
Europe 03:00–09:30    | 124     | 77.42   | 50.00     | 50.00     | 50.00     | 82.26          | 79.03   
RTH 09:30–16:00       | 92      | 94.57   | 96.74     | 96.74     | 96.74     | 100.00         | 100.00  


## Results

表内 `WF ΔR² %` 相对逐日 training-mean baseline；正数仅表示该样本下相对误差更低，不表示可交易或因果。

In [10]:
focus_names = {
    'Net GEX proxy, $bn', 'log1p gross gamma',
    'log1p gross Charm 5m', 'log1p gross Vanna 1vol',
    'Side-aligned zero-gamma distance', 'Put ratio − Call ratio',
}
focus = [row for row in payload['primary_ablation'] if row['feature'] in focus_names]
print(compact_table(
    focus,
    ['session','horizon','feature','n','days','spearman','low_tertile_mean_points','high_tertile_mean_points','wf_delta_r2_pct','wf_labels','wf_test_days'],
))

session | horizon | feature                          | n   | days | spearman | low_tertile_mean_points | high_tertile_mean_points | wf_delta_r2_pct | wf_labels | wf_test_days
--------+---------+----------------------------------+-----+------+----------+-------------------------+--------------------------+-----------------+-----------+-------------
RTH     | 15m     | Net GEX proxy, $bn               | 70  | 7    | 0.10     | -1.06                   | 2.03                     | -13.08          | 36        | 3           
RTH     | 15m     | log1p gross gamma                | 72  | 7    | -0.22    | 1.92                    | -1.50                    | 1.72            | 37        | 3           
RTH     | 15m     | log1p gross Charm 5m             | 72  | 7    | -0.14    | -0.31                   | -2.40                    | -3.37           | 37        | 3           
RTH     | 15m     | log1p gross Vanna 1vol           | 72  | 7    | -0.17    | 1.22                    | -0.99               

In [11]:
summary = {(row['cohort'], row['session']): row for row in payload['cohort_summary']}
assert summary[('All ES origins','RTH')]['origins'] == 129
assert summary[('All ES origins','GTH')]['origins'] == 392
assert summary[('Directional headline','RTH')]['origins'] == 92
assert summary[('Directional headline','GTH')]['origins'] == 250
assert [summary[('Directional headline','RTH')][f'labels_{h}m'] for h in (15,30,60)] == [74,71,60]
assert [summary[('Directional headline','GTH')][f'labels_{h}m'] for h in (15,30,60)] == [245,246,239]
assert all(row['wf_test_days'] <= 4 for row in payload['primary_ablation'])
print('VALIDATED: causal cutoff, cohort counts, date-block split, and shadow-only tables.')

VALIDATED: causal cutoff, cohort counts, date-block split, and shadow-only tables.


## Takeaways

1. RTH 有可审计标签，但 7 个日期不足以调生产硬阈值。
2. GTH gross Greek 覆盖较差，Europe 段尤其明显；缺失不能补零。
3. Charm 仅作幅度/质量 gate，Call+/Put− GEX 不等于 dealer sign。
4. 先积累 20+20 完整会话与每 horizon 100 个非重叠 forward 标签，再做冻结参数的 date-block CI。
5. 同刻 0DTE/1DTE ask→bid 配对已经可做，但 >=13:00 ET 仅 8/7/6 个 outcome 且 CI 全跨 0；不得上线 13:00 硬切换。
6. 下一步补真实 fill/fee、vertical-spread 双腿、ATM IV term gap 与 Theta/Gamma；本研究不能解释账户 P&L。